# Reviewing Roman image cutouts

## Goal and Learning Goals

Use a **specific real MAST dataset** to learn the complete workflow: make a bounded
request, understand the YAML, verify cutout positions against parent images and an
optional independent finding chart, review every quality decision, and change
screening/display settings without remaking native cutouts.

## Introduction and Setup

This notebook uses Gladius to review a selected MAST image. Both the local and
Nexus tutorials follow this workflow and require a MAST credential file. The
Python environment running the notebook needs `roman-gladius` installed; the
sibling requirements file lists the dependencies.

**Real data only; no synthetic fallback.** You must choose an accessible dataset
and an explicit credential-file path. A full calibrated detector image is downloaded,
even for a small cutout. Its advertised size is checked before running; this is
not a hard streaming quota. Keep restricted coordinates, images and outputs private.
This notebook remains separate from RTD and is committed with cleared outputs.


### Choose credentials, dataset and targets

1. Create/use a token from [MAST API tokens](https://auth.mast.stsci.edu/tokens).
   Your chosen file must contain `MAST_API_TOKEN=...`; never paste the token here.
2. Choose an observation in [MAST](https://mast.stsci.edu/) that your account can
   access, and obtain its Roman `fileSetName` dataset identifier from its metadata.
   Use a known observation from your science sample, not an arbitrary all-archive search.
   The cell below refuses blank IDs and wildcard/range syntax.
3. Optionally supply a CSV with `id,ra,dec` in ICRS degrees (or zero-based `id,x,y`).
   Extra columns are preserved. A sky-coordinate target does not itself identify a star.

With `TARGET_CSV=None`, the dataset's archive centre is used only as a clearly
labelled practice location. For your actual science, use your catalogue and finding chart.
Set the maximum full-parent size you are prepared to download.


In [ ]:
from pathlib import Path

ENV_FILE = None
# ENV_FILE = Path("/path/to/.env")  # Uncomment and replace with your environment file path.
DATASET_ID = None
# DATASET_ID = "your-exact-Roman-fileSetName"
WORKSPACE = Path.cwd() / "gladius-archive-review"
TARGET_CSV = None
# TARGET_CSV = Path("/path/to/targets.csv")
CUTOUT_SIZE = 150
MAX_PARENT_MIB = 512
RUN_GUI = False

# The scheme, hostname, and port in your JupyterLab address, without its path.
NOTEBOOK_ORIGIN = "http://localhost:8888"


In [ ]:
import os
from dotenv import dotenv_values

if ENV_FILE is None:
    raise ValueError("Set ENV_FILE to the path of your local credential file, then rerun this cell.")
ENV_FILE = Path(ENV_FILE).expanduser().resolve()
if not ENV_FILE.is_file():
    raise FileNotFoundError("ENV_FILE must point to an existing file; no archive request was made.")
credentials = dotenv_values(ENV_FILE, interpolate=False)
if not (credentials.get("MAST_API_TOKEN") or "").strip():
    raise ValueError("The selected file must contain a nonempty MAST_API_TOKEN.")
os.environ["MAST_API_TOKEN"] = credentials["MAST_API_TOKEN"]
del credentials

WORKSPACE = Path(WORKSPACE).expanduser().resolve()
if not isinstance(CUTOUT_SIZE, int) or isinstance(CUTOUT_SIZE, bool) or CUTOUT_SIZE < 1:
    raise ValueError("CUTOUT_SIZE must be a positive integer.")
if not isinstance(MAX_PARENT_MIB, (int, float)) or MAX_PARENT_MIB <= 0:
    raise ValueError("MAX_PARENT_MIB must be positive.")
print("Credential file loaded. Token is not displayed.")


## Prepare the exact metadata request, before sending it

The first request below is restricted by `fileSetName` **and** calibrated Level 2.
It is not a coordinate cross-match across Roman. `limit=2` is a response/truncation
check, **not** a guarantee that a broad query would be cheap. Exact criteria provide
the bound. Printing this dictionary does not contact MAST.


In [ ]:
if not isinstance(DATASET_ID, str) or not DATASET_ID.strip():
    raise ValueError("Set DATASET_ID to one exact Roman fileSetName. No MAST query was sent.")
DATASET_ID = DATASET_ID.strip()
if not all(c.isalnum() or c in "_-" for c in DATASET_ID):
    raise ValueError("Use an exact dataset ID, without wildcards, lists or ranges.")
metadata_request = {"fileSetName": DATASET_ID, "productLevel": "2"}
print("Metadata criteria:", metadata_request)
print("Response limit: 2; expected exact match: 1 dataset. No download yet.")


### Send that bounded metadata request

This cell sends the request just printed, lists calibrated products from that
single dataset and selects the smallest known-size calibrated image from it.
There is no claim that this is the smallest image in MAST. No image is downloaded.


In [ ]:
import json
import math
import numpy as np
from astropy.table import Table
from IPython.display import display
from astroquery.mast import MastMissions
from gladius import RunConfig, Target, read_targets_csv, run

missions = MastMissions(mission="roman")
missions.TIMEOUT = 30
missions.login(token=os.environ["MAST_API_TOKEN"], store_token=False)
observations = missions.query_criteria(
    **metadata_request, limit=2,
    select_cols=["fileSetName", "ra", "dec", "detector", "optical_element", "exposure_start_time"])
if len(observations) != 1 or str(observations[0]["fileSetName"]) != DATASET_ID:
    raise ValueError("Expected exactly the requested dataset. Narrow/fix DATASET_ID; no download made.")
products = missions.filter_products(missions.get_unique_product_list(observations), file_suffix="_cal")
candidates = []
for product in products:
    name = str(product["filename"])
    try:
        size = float(product["size"])
    except (ValueError, TypeError, KeyError):
        continue
    if name.endswith("_cal.asdf") and math.isfinite(size) and size > 0 and size == int(size):
        candidates.append((int(size), name))
if not candidates:
    raise ValueError("No calibrated 2D product with a known size in this dataset; no download made.")
parent_bytes, product_filename = min(candidates)
if parent_bytes > MAX_PARENT_MIB * 1024**2:
    raise ValueError(f"Advertised parent size is {parent_bytes/1024**2:.1f} MiB; exceeds MAX_PARENT_MIB.")
display(observations)
print("Exact product:", product_filename)
print("Advertised download:", round(parent_bytes/1024**2, 2), "MiB")


## Understand the configuration before running

**SCI** is the science-pixel array, **ERR** its per-pixel uncertainty, and **DQ**
contains integer data-quality flags. **WCS** means the coordinate model mapping
pixels to sky positions. **ICRS** is the celestial reference frame used for the
RA/Dec coordinates here. ASDF and FITS are the saved file formats.

`RunConfig` is the Python form of the YAML. Omitted keys get defaults; misspelled
keys fail validation. Creating it and printing YAML do **not** query MAST.

| Section | What you choose | What this example does |
| --- | --- | --- |
| `run.name` | Name of the working project. | New settings become numbered revisions of this name. |
| `targets` | IDs and ICRS RA/Dec in **degrees**, or zero-based detector x/y. | Each selected parent is tried against every target. A catalogue position is a prediction, not a detected star. |
| `inputs` | Local directory/pattern or MAST criteria. | Explicit input set below; the query is printed before any request. |
| `selection` | UTC exposure-start dates `[start,end)`, detector numbers and filters. | Use dates/detectors/filters for your own sample; this exact-dataset example needs no date range. |
| `extraction` | Square side in pixels and boundary handling. | `pad` retains the requested size with unavailable pixels marked at detector edges; targets not on a parent detector are skipped. |
| `astrometry` | Supplied or corrected WCS; numerical checks. | Numerical round trips check internal consistency, **not** whether the image points at the right stars. |
| `quality` | Autocorrelation tile, lag radius and threshold. | A provisional elongation screen; a failed screen still saves the native cutout. |
| `stacks` | Independent median, GIF and RGB choices. | Display products use eligible cutouts; native images are for measurements. |
| `storage` | Output location, ASDF/FITS and parent retention. | Retain parents for context images; local original files are never deleted. |
| `execution` | Reuse, continue after errors, lock timeout. | Reuse enabled; stop on errors. Inspect manifests even if a run returns. |

Use the GUI below to inspect help and build YAML, or the
[full configuration reference](https://roman-gladius.readthedocs.io/en/latest/configuration.html).
Changing dates/queries refreshes discovery. Changing quality or display settings
reuses verified native cutouts. `run(config, refresh_inputs=True)` refreshes MAST
membership; `fresh=True` deliberately bypasses reuse. Completed revisions stay intact.


In [ ]:
if TARGET_CSV is None:
    targets = [Target(id="archive-centre-not-a-star-id",
                      ra=float(observations[0]["ra"]), dec=float(observations[0]["dec"]))]
else:
    targets = read_targets_csv(Path(TARGET_CSV).expanduser())
config = RunConfig(
    run={"name": "archive-field-review"},
    targets=targets,
    inputs={"mode": "mast", "mast_criteria": {"fileSetName": DATASET_ID},
            "product_filename": product_filename, "query_limit": 2, "mast_radius_arcmin": 6},
    extraction={"size": CUTOUT_SIZE, "edge": "pad"},
    astrometry={"mode": "validate", "roundtrip_tolerance_pixels": 0.01},
    quality={"autocorrelation_limit": 2.6, "tile_size": 64, "radius": 15,
             "exclude_from_stacks": True},
    stacks={"enabled": False, "gif_enabled": False, "rgb_enabled": False},
    storage={"output_directory": "output", "retain_full_images": True,
             "formats": ["asdf", "fits"]},
).resolved(WORKSPACE)


### Inspect the pipeline queries and full YAML

`mast_queries(config)` is an offline function. It shows the **actual** criteria
Gladius will send, not a guessed example. This tutorial asserts that every query
still contains the exact dataset. Sky targets also add a cone; the dataset bound
is retained. The cone selects candidate images, not proof that each target lies
on that detector; the saved cutout or its `target_off_detector` skip answers that.

For a larger science run, use program/dataset constraints plus date bounds,
detectors/filters and a sensible cone radius. Inspect every generated query before
running. Advanced `mast_criteria` can override generated date/filter/position
criteria, so do not copy arbitrary overrides. A query limit only detects possible
truncation; it is not a bound on the archive's search work or a download budget.

`mast_criteria` needs MAST field names (for example `fileSetName`, `productLevel`,
`optical_element`, `exposure_start_time`), not `RunConfig` field names. The helper
translates ordinary `selection` fields for you. See the
[Roman criteria docs](https://astroquery.readthedocs.io/en/stable/mast/mast_missions.html#mission-criteria-queries).


In [ ]:
import yaml
from gladius.inputs import mast_queries

queries = mast_queries(config)
assert all(query.get("fileSetName") == DATASET_ID for query in queries)
print("Pipeline query count:", len(queries))
for query in queries:
    print(query)
print("Exact downloaded product:", config.inputs.product_filename)
print(yaml.safe_dump(config.model_dump(mode="json"), sort_keys=False))
WORKSPACE.mkdir(parents=True, exist_ok=True)
(WORKSPACE / "review-config.yaml").write_text(
    yaml.safe_dump(config.model_dump(mode="json"), sort_keys=False))


## Download that parent and make native cutouts

**This cell accesses the image bytes.** It may repeat the bounded metadata query
shown above before downloading. The exact product is pinned; it cannot silently
expand to every Roman image. The full image is retained for context previews.
The same image is opened once for all targets. Nothing deletes local originals.


In [ ]:
manifest_path = run(config, progress=print)
manifest = json.loads(manifest_path.read_text())
print("Manifest:", manifest_path)
print("Work performed:", manifest["stage_counts"])


## Check every attempted parent/target pair

The audit table includes **complete, skipped and failed** entries. `accepted=False`
means a complete native cutout was flagged by the optional screen; it does not
mean the cutout was discarded. `target_off_detector` is different: the target is
not on that detector. Errors are failures, not quality judgements.

Inspect the table before trusting the previews. If you expected more parent images,
compare the input list/query and date/filter selection; quality screening does not
remove a parent from discovery. A complete run does not prove a star is centred
or that every catalogue target was on every detector.


In [ ]:
from astropy.table import Table
from IPython.display import display
from gladius.inspection import audit_rows, contact_sheets, parent_preview

# A Python list of dictionaries: one row for every attempted input/target pair.
review = audit_rows(manifest_path)
audit = Table(rows=review)
display(audit[["target", "parent", "filter", "status", "accepted", "display_eligible", "axis_ratio", "reason"]])
display(audit[["target", "parent", "padding_fraction", "do_not_use_fraction", "saturated_fraction"]])
print("Run status:", manifest["status"])
print("Input images used:", len(manifest["sources"]))
print("Parents rejected by date/detector/filter:", manifest["selection_rejections"])
print("Recorded errors:", manifest["errors"])


## What was cut out, and is it the right place?

Each parent preview has **requested cutout boxes** and **predicted target crosses**.
It shows the original detector orientation; N/E arrows come from the saved WCS.
Boxes may cross an edge when padding is allowed. Purple areas in cutout previews
are unavailable pixels, not empty sky.

Compare the star pattern and target location against a finding chart. The cross
is not a fitted centroid. An internally consistent but incorrect WCS can put it
on the wrong star. An independent finding chart or known reference sources are
needed to test absolute pointing; a parent and its own cutout are not independent.

The optional reference-image cell below makes a side-by-side visual comparison
possible by displaying your chart in the same notebook. It does not rotate,
register, cross-match, or automatically certify a match. Match scale/orientation
using the chart's labels and these N/E arrows before judging the star pattern.


In [ ]:
# Show every retained/available parent, once, with all its completed cutout boxes.
parent_ids = sorted({entry["parent_sha256"] for entry in manifest["entries"].values()
                     if entry["status"] == "complete"})
for parent_id in parent_ids:
    display(parent_preview(manifest_path, parent_id, width=800))


In [ ]:
from PIL import Image

FINDING_CHART = None
# FINDING_CHART = Path("/path/to/finding-chart.png")  # Optional independent chart.
if FINDING_CHART is not None:
    with Image.open(FINDING_CHART) as chart:
        reference = chart.convert("RGB")
        reference.thumbnail((900, 900))
        display(reference)


## Review ALL cutouts, including the flagged ones

The following pages include every complete cutout; none are hidden because the
screen rejected them. Compare **KEEP** and **FLAG** thumbnails to look for both
false exclusions and missed defects. The table above retains every full filename.

- A cross marks the requested position. N/E arrows retain the native orientation.
- The gold rectangle is the central region used by autocorrelation. Damage outside
  it can be missed: the screen is not a whole-image quality classifier.
- The score is an autocorrelation long/short axis ratio: near 1 is roughly round;
  higher is more elongated. It is not a fitted stellar PSF shape.
- The same target/filter/unit group shares one asinh brightness stretch, so changes
  are visible. Different groups have their own stretch; displayed brightness is not
  a calibrated cross-filter comparison.
- Padding and bad/saturated pixel fractions help identify missing coverage or DQ
  problems. Inspect artifacts, centring and background by eye too.

`insufficient_tile`, `insufficient_signal` and `degenerate_acf` are failed
measurements, **not** evidence of trailing. A smaller cutout than `tile_size` will
always fail the enabled screen. Do not solve that by blindly raising its threshold.


In [ ]:
# Twelve images per PNG page; iterate every page to review all complete cutouts.
review_directory = WORKSPACE / "review" / config.run.name / manifest_path.parent.name
review_directory.mkdir(parents=True, exist_ok=True)
for page_number, sheet in enumerate(contact_sheets(manifest_path), start=1):
    path = review_directory / f"all-cutouts-{page_number:03d}.png"
    sheet.save(path)
    display(sheet)
print("Review PNGs:", review_directory)


## Tune the screen without downloading or cutting again

Use the audit and thumbnails to choose a threshold. A value such as **2.6** is an
example, not a calibrated recommendation for Roman. Increase it to tolerate more
elongation; decrease it to flag more images. `None` disables the screen. A score
**at or above** the threshold is flagged. `tile_size` is the side length, in pixels, of the central square used for the
screen. `radius` is the largest pixel shift included when measuring the shape of
that square's autocorrelation (its similarity to shifted copies of itself).
These choices also affect the score; changing them measures a different region/shape and needs visual checking.

When `exclude_from_stacks=True`, flagged cutouts are omitted from display products.
They remain available for inspection and scientific decisions. Setting it to False
includes them in displays; it does not clear their flags. There is currently no
per-exposure manual keep/exclude override in the tool: use these settings and
explicit input/date selections, and retain your reasons alongside the run.

The next cell creates a revision of the same named run. With the native cache
available, it does not repeat extraction. The revision manifest has the **current**
quality decision; reused image headers retain their original creation provenance.


In [ ]:
# Edit these values after inspecting the first audit and contact sheets.
NEW_ACF_LIMIT = 2.0  # Demonstration only; choose based on representative images.
NEW_TILE_SIDE = config.quality.tile_size
NEW_LAG_RADIUS = config.quality.radius

settings = config.model_dump(mode="json")
settings["quality"].update(autocorrelation_limit=NEW_ACF_LIMIT,
                           tile_size=NEW_TILE_SIDE, radius=NEW_LAG_RADIUS)
review_config = RunConfig.model_validate(settings)
old_manifest_path = manifest_path
manifest_path = run(review_config, progress=print)
manifest = json.loads(manifest_path.read_text())
print("Previous revision:", old_manifest_path.parent)
print("Current revision:", manifest_path.parent)
print("Work performed:", manifest["stage_counts"])
# With the cache intact, 'extracted' is 0. Original revision files remain unchanged.
audit = Table(rows=audit_rows(manifest_path))
display(audit[["target", "parent", "filter", "accepted", "display_eligible", "axis_ratio", "reason"]])
review_directory = WORKSPACE / "review" / config.run.name / manifest_path.parent.name
review_directory.mkdir(parents=True, exist_ok=True)
for page_number, sheet in enumerate(contact_sheets(manifest_path), start=1):
    sheet.save(review_directory / f"all-cutouts-{page_number:03d}.png")
    display(sheet)
config = review_config


## Add or rebuild display products without repeating extraction

A **median stack** combines aligned exposure pixels. A **GIF** plays individual
aligned exposures in time order. An **RGB composite** maps three independently
normalized filter medians to colours. They are different products and can be
selected independently. A one-exposure median is just that exposure; it does not
provide a noise-reduction demonstration.

Adding a composite reuses native cutouts and existing independent previews.
RGB needs three eligible filters for the same target; the example below prints
missing-filter reasons instead of pretending that one filter is a colour image.
For a real composite, select actual blue/green/red filters present in your input
inventory and use a bounded dataset/program/date search that includes them.

Stacks/GIFs/composites are for inspection. They resample pixels and do not propagate
uncertainties or perform PSF/photometric matching. Use native cutouts for flux or
position measurements. Check the display manifest's exact contributors and exclusions.


In [ ]:
settings = config.model_dump(mode="json")
settings["stacks"].update(enabled=True, gif_enabled=True, rgb_enabled=True,
                          blue_filter="F062", green_filter="F184", red_filter="F213")
config = RunConfig.model_validate(settings)
manifest_path = run(config, progress=print)
manifest = json.loads(manifest_path.read_text())
display_manifest = json.loads((manifest_path.parent / "stack-manifest.json").read_text())
print("Native work:", manifest["stage_counts"])
for category in ("products", "animations", "composites", "composite_skips", "excluded"):
    print(category, display_manifest[category])


In [ ]:
from astropy.io import fits
from IPython.display import Image as DisplayImage
from gladius.inspection import image_preview

# Median images: display stretch only; the FITS science values stay unchanged.
for product in display_manifest["products"]:
    path = manifest_path.parent / product["path"]
    with fits.open(path) as hdul:
        print("Median:", product["target"], product["filter"], "inputs:", len(product["inputs"]))
        display(image_preview(hdul[0].data))
        coverage = hdul["COVERAGE"].data
        print("Contributing exposures per output pixel:", int(coverage.min()), "to", int(coverage.max()))

# The GIF is individual aligned exposures, NOT the median shown above.
for animation in display_manifest["animations"]:
    print("GIF:", animation["target"], animation["filter"], "frames:", len(animation["frames"]))
    display(DisplayImage(filename=str(manifest_path.parent / animation["path"])))

# RGB has independently stretched filter channels; it is not physical colour.
for composite in display_manifest["composites"]:
    print("RGB:", composite["target"], composite["channel_filters"])
    with Image.open(manifest_path.parent / composite["path"]) as image:
        display(image.resize((300, 300), Image.Resampling.NEAREST))


## Build and edit the same configuration in the GUI

Set `RUN_GUI=True` in the setup cell, then run the next cell. It loads the current
reviewed configuration and provides field help, YAML import/export and **Make**.
You can use it to make YAML without starting a run. The GUI runs in this kernel.

For a local notebook, `NOTEBOOK_ORIGIN` identifies the JupyterLab address allowed
to embed the interface. It must match your browser’s scheme, hostname, and port.
The interface runs on a separate localhost port.
Changing settings in the GUI does not automatically change the Python `config`
variable: download its YAML and load it with `load_config(path)` to use it in code.
Stop the server below before restarting it with a different configuration.


In [ ]:
if RUN_GUI:
    import threading
    from IPython.display import IFrame, display
    from gladius.web import GuiServer

    if "gui_thread" not in globals() or not gui_thread.is_alive():
        gui_server = GuiServer(WORKSPACE, port=0, iframe_origin=NOTEBOOK_ORIGIN, initial_config=config)
        gui_thread = threading.Thread(target=gui_server.serve_forever, daemon=True)
        gui_thread.start()
    gladius_ui_url = gui_server.url + "/"
    print("Open the GUI in a separate tab:", gladius_ui_url)
    display(IFrame(src=gladius_ui_url, width="100%", height=650))
else:
    print("GUI skipped. Set RUN_GUI=True in the settings cell to explore it.")


### Stop the GUI when finished

Wait for any job to finish first. Closing an iframe does not stop the server.


In [ ]:
if "gui_thread" in globals() and gui_thread.is_alive():
    gui_server.shutdown()
    gui_server.server_close()
    gui_thread.join(timeout=5)
    print("Embedded GUI stopped.")


## Summary and Next Steps

You bounded the query before requesting data, saved a reproducible YAML, inspected
parents and every cutout, and changed quality/display choices using saved pixels.
The audit distinguishes bad-screen decisions, missing coverage and processing errors.
A one-parent example cannot demonstrate real multi-exposure noise reduction or a
three-filter colour composite; expand the input selection deliberately to do those.

For your science sample, use known targets plus an independently checked finding
chart. Choose explicit program/dataset/date/filter bounds, print `mast_queries`,
then inspect the selected inventory and download sizes before making many cutouts.
Use `run(config, refresh_inputs=True)` to look for new archive members; known MAST
product identities are trusted. `fresh=True` intentionally downloads/recomputes.
Gladius currently has no per-exposure manual quality override or automatic cache
quota. Do not claim that an elongation threshold certifies scientific image quality.

## Additional Resources

- [Configuration and reuse rules](https://roman-gladius.readthedocs.io/en/latest/configuration.html)
- [Outputs and manifests](https://roman-gladius.readthedocs.io/en/latest/products.html)
- [Scientific limits](https://roman-gladius.readthedocs.io/en/latest/science.html)
- [MAST authentication](https://auth.mast.stsci.edu/tokens)

## About This Notebook and Citations

Authors: Gladius contributors. Updated 2026-11-02. This is an independent community
tutorial.
Dependencies: Gladius, Astroquery/MAST, Astropy, ASDF, NumPy and Pillow. Cite Gladius
using the repository CITATION.cff and cite the underlying mission data/software
as appropriate for your analysis.

Offline synthetic archive fixtures exercise this workflow without archive access.
A full real-MAST execution of this revised notebook still requires an accessible
DATASET_ID and explicitly selected credential file. Clear outputs before committing
or sharing, especially when observations or pointing metadata are restricted.
